# Classical Baseline Sanity Test — High SNR / Zero Noise

**উদ্দেশ্য:** একই Y matrix দিয়ে দুটো জিনিস verify করা —

1. **GT heatmap → AoA/AoD recovery ঠিক আছে কিনা** (coordinate mapping check — Bug 4 এখানেই ধরা পড়েছিল)
2. **Classical state-of-the-art methods** একই Y-তে কত RMSE / Pd দেয়

### Pipeline
```
true (ψ, φ) ──► Y matrix (16×16) ──► GT heatmap (256×256) ──► blob ──► (ψ̂, φ̂)   [check #1]
                     │
                     └──► classical DFT / DFT-SIC / MUSIC ──► (ψ̂, φ̂)             [check #2]
```

Zero noise + 25 dB দুটোতেই চালাবে। Zero noise-এ error ≈ 0 হওয়া উচিত।

In [ ]:
# Cell 1 — Setup
import importlib, subprocess, sys
try:
    import cv2
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'opencv-python-headless'], check=True)
    import cv2

import numpy as np, math, time
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment
from scipy.ndimage import maximum_filter

np.random.seed(0)
print('✅ Ready')

In [ ]:
# Cell 2 — Physics (paper-exact) + sample generator
class _H(np.ndarray):
    @property
    def H(self): return self.conj().transpose()

def ev(n, ang):
    return ((1/np.sqrt(n))*np.exp(-1j*np.pi*np.cos(ang)*np.arange(n))).reshape(-1,1)

def make_F(P, nt):
    ph = np.arccos((1/np.pi)*np.angle(np.exp( 1j*(2*np.pi/P)*np.arange(P))))
    F = np.zeros((nt, P), complex)
    for i, a in enumerate(ph): F[:, i] = ev(nt, a).ravel()
    return F

def make_W(Q, nr):
    ph = np.arccos((1/np.pi)*np.angle(np.exp(-1j*(2*np.pi/Q)*np.arange(Q))))
    W = np.zeros((nr, Q), complex)
    for i, a in enumerate(ph): W[:, i] = ev(nr, a).ravel()
    return W

def gen_channel(nr, nt, phi_l, psi_l, al):
    Hm = np.zeros((nr, nt), complex)
    for a, ph, ps in zip(al, phi_l, psi_l):
        Hm += a * (ev(nr, ps) * ev(nt, ph).view(_H).H)
    return np.sqrt(nt*nr) * Hm

def gen_points(L, delta=np.pi/6, max_try=20000):
    pts = []
    for _ in range(max_try):
        if len(pts) == L: break
        x, y = np.random.uniform(0, np.pi), np.random.uniform(0, np.pi)
        if all(math.hypot(x-p[0], y-p[1]) >= delta for p in pts): pts.append((x, y))
    if len(pts) < L: raise RuntimeError('point placement failed')
    return pts

def gen_gt(phi_l, psi_l, M=256, sigma=0.07):
    op = np.mod( np.pi*np.cos(phi_l), 2*np.pi)   # AoD  → column axis
    oq = np.mod(-np.pi*np.cos(psi_l), 2*np.pi)   # AoA  → row    axis
    margin = 3*sigma
    ax = np.linspace(-margin, 2*np.pi+margin, M, endpoint=False)
    Wp, Wq = np.meshgrid(ax, ax)
    c = 1/(2*np.pi*sigma**2)
    G = sum(c*np.exp(-((Wp-o)**2 + (Wq-q)**2)/(2*sigma**2)) for o, q in zip(op, oq))
    return G.astype(np.float32)

def make_sample(L=3, SNR=25, P=16, nt=16, sigma=0.07, M=256, noiseless=False):
    """Returns Y (Q×P complex), GT heatmap, true psi, true phi, alphas."""
    Q, nr = P, nt
    F, W = make_F(P, nt), make_W(Q, nr)
    al = (np.sqrt(1/L)/np.sqrt(2))*(np.random.randn(L) + 1j*np.random.randn(L))
    al = al[np.argsort(-np.abs(al))]
    pts  = gen_points(L)
    phi_l = np.array([p[0] for p in pts])
    psi_l = np.array([p[1] for p in pts])
    Hm = gen_channel(nr, nt, phi_l, psi_l, al)
    if noiseless:
        Z = np.zeros((Q, P), complex)
    else:
        var = 10**(-SNR/10); s = np.sqrt(var/2)
        Z = s*(np.random.randn(Q, P) + 1j*np.random.randn(Q, P))
    Y = (W.view(_H).H @ Hm) @ F + Z
    gt = gen_gt(phi_l, psi_l, M, sigma)
    return Y, gt, psi_l, phi_l, al

print('✅ Physics ready')

In [ ]:
# Cell 3 — GT heatmap → angles (blob detection), same as NN pipeline
def get_detector():
    p = cv2.SimpleBlobDetector_Params()
    p.filterByColor = True; p.blobColor = 255
    p.minThreshold = 0;     p.maxThreshold = 255
    p.filterByArea = True;  p.minArea = 1; p.maxArea = 1000
    p.filterByCircularity = p.filterByConvexity = p.filterByInertia = False
    return cv2.SimpleBlobDetector_create(p)

DET = get_detector()

def peaks_from_img(img2d, L):
    im = cv2.normalize(img2d, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    kps = DET.detect(im)
    if not kps: return np.zeros((0, 2))
    co = np.array([k.pt for k in kps])                  # (x=col, y=row)
    am = np.array([im[min(int(round(k.pt[1])), im.shape[0]-1),
                      min(int(round(k.pt[0])), im.shape[1]-1)] for k in kps])
    return co[np.argsort(-am)[:L]]

def peaks2angles(peaks, sigma=0.07, M=256):
    """Inverse of gen_gt's axis mapping."""
    if len(peaks) == 0: return np.array([]), np.array([])
    margin = 3*sigma; ext = 2*np.pi + 2*margin
    f = -margin + (peaks.T/M)*ext                       # f[0]=col→ωφ, f[1]=row→ωψ
    f = np.where(f > np.pi, f - 2*np.pi, f)             # wrap to [-π, π]
    psi = np.arccos(np.clip(-f[1]/np.pi, -1, 1))
    phi = np.arccos(np.clip( f[0]/np.pi, -1, 1))
    return psi, phi

print('✅ Blob → angle ready')

In [ ]:
# Cell 4 — Classical methods
#
#   Y[q,p] = 2D-DFT of  x[m,n] = Σ_l α_l · exp(-j·m·u_l) · exp(+j·n·v_l)
#     u = π·cos(ψ)  (AoA),  v = π·cos(φ)  (AoD)
#   ⇒  x = ifft2(Y);  zero-pad + fft2  ⇒  fine frequency grid (this IS DFT-CEA)

def _wrap(a):
    return (a + np.pi) % (2*np.pi) - np.pi

def _bin2angles(k1, k2, N):
    u = _wrap(-2*np.pi*np.asarray(k1)/N)   # π cos ψ
    v = _wrap( 2*np.pi*np.asarray(k2)/N)   # π cos φ
    psi = np.arccos(np.clip(u/np.pi, -1, 1))
    phi = np.arccos(np.clip(v/np.pi, -1, 1))
    return psi, phi

def _top_local_maxima(mag, L):
    mx  = maximum_filter(mag, size=3, mode='wrap')
    idx = np.argwhere(mag == mx)
    if len(idx) == 0: return np.zeros((0, 2), int)
    vals = mag[idx[:, 0], idx[:, 1]]
    return idx[np.argsort(-vals)[:L]]

# ── Method 1: plain 2D-DFT peak search (DFT-CEA style) ──
def classical_dft(Y, L, NDFT=1024):
    x = np.fft.ifft2(Y)
    X = np.fft.fft2(x, s=(NDFT, NDFT))
    idx = _top_local_maxima(np.abs(X), L)
    if len(idx) < L: return np.array([]), np.array([])
    return _bin2angles(idx[:, 0], idx[:, 1], NDFT)

# ── Method 2: 2D-DFT + successive interference cancellation (CLEAN / TSDCE spirit) ──
def classical_dft_sic(Y, L, NDFT=1024):
    r = np.fft.ifft2(Y).copy()
    Q, P = r.shape
    m = np.arange(Q).reshape(-1, 1); n = np.arange(P).reshape(1, -1)
    ks1, ks2 = [], []
    for _ in range(L):
        X = np.fft.fft2(r, s=(NDFT, NDFT))
        k = np.unravel_index(np.argmax(np.abs(X)), X.shape)
        ks1.append(k[0]); ks2.append(k[1])
        u = _wrap(-2*np.pi*k[0]/NDFT)
        v = _wrap( 2*np.pi*k[1]/NDFT)
        a = X[k] / (Q*P)                                    # LS amplitude
        r -= a * np.exp(-1j*m*u) * np.exp(1j*n*v)           # cancel this path
    return _bin2angles(ks1, ks2, NDFT)

# ── Method 3: 2D-MUSIC with spatial smoothing (single snapshot) ──
def classical_music(Y, L, sub=12, Ngrid=361):
    x = np.fft.ifft2(Y)
    Q, P = x.shape
    sm, sn = sub, sub
    snaps = []
    for i in range(Q-sm+1):
        for j in range(P-sn+1):
            snaps.append(x[i:i+sm, j:j+sn].ravel())
    S = np.array(snaps).T                                   # (sm*sn, n_snap)
    R = (S @ S.conj().T) / S.shape[1]
    w, V = np.linalg.eigh(R)
    En = V[:, :-L]                                          # noise subspace

    ug = np.linspace(-np.pi, np.pi, Ngrid)
    vg = np.linspace(-np.pi, np.pi, Ngrid)
    mm = np.arange(sm).reshape(-1, 1); nn = np.arange(sn).reshape(-1, 1)
    Am = np.exp(-1j*mm*ug)                                  # (sm, Ngrid)
    An = np.exp( 1j*nn*vg)                                  # (sn, Ngrid)

    spec = np.zeros((Ngrid, Ngrid))
    EnH = En.conj().T
    for iu in range(Ngrid):
        Ablk = (Am[:, iu].reshape(-1, 1) * An.T[:, :].T)    # not used; explicit below
        A = np.einsum('i,jk->ijk', Am[:, iu], An).reshape(sm*sn, Ngrid)
        pr = EnH @ A
        spec[iu, :] = 1.0/np.maximum(np.sum(np.abs(pr)**2, axis=0), 1e-12)

    idx = _top_local_maxima(spec, L)
    if len(idx) < L: return np.array([]), np.array([])
    u = ug[idx[:, 0]]; v = vg[idx[:, 1]]
    psi = np.arccos(np.clip(u/np.pi, -1, 1))
    phi = np.arccos(np.clip(v/np.pi, -1, 1))
    return psi, phi

print('✅ Classical methods ready (DFT, DFT-SIC, MUSIC)')

In [ ]:
# Cell 5 — Metric (paper-exact): Hungarian match, 1° threshold
def evaluate(est_psi, est_phi, true_psi, true_phi, max_deg=1.0):
    """Returns (n_detected_sources, n_total_sources, good_angle_errors_deg)."""
    L = len(true_psi)
    if len(est_psi) < L: return 0, L, []
    gt = np.stack([true_psi, true_phi], 1)
    es = np.stack([est_psi[:L], est_phi[:L]], 1)
    d  = np.linalg.norm(gt[:, None] - es[None], axis=2)
    r, c = linear_sum_assignment(d)
    ndet, good = 0, []
    for i, j in zip(r, c):
        dpsi = np.degrees(np.angle(np.exp(1j*gt[i, 0]) * np.exp(-1j*es[j, 0])))
        dphi = np.degrees(np.angle(np.exp(1j*gt[i, 1]) * np.exp(-1j*es[j, 1])))
        if abs(dpsi) <= max_deg and abs(dphi) <= max_deg:
            ndet += 1
            good += [dpsi, dphi]
    return ndet, L, good

print('✅ Metric ready')

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 6 — CHECK #1: GT heatmap → angle recovery (coordinate test)
# ══════════════════════════════════════════════════════════════
np.random.seed(7)
L = 3

for tag, kw in [('ZERO NOISE', dict(noiseless=True)), ('SNR = 25 dB', dict(SNR=25))]:
    Y, gt, psi_t, phi_t, al = make_sample(L=L, P=16, nt=16, **kw)

    pk = peaks_from_img(gt, L)
    psi_e, phi_e = peaks2angles(pk)
    ndet, ntot, good = evaluate(psi_e, phi_e, psi_t, phi_t)

    print(f'\n{"═"*62}')
    print(f'  {tag}  —  GT heatmap → blob → angles')
    print(f'{"═"*62}')
    print(f'  true ψ (AoA deg): {np.degrees(psi_t).round(3)}')
    print(f'  true φ (AoD deg): {np.degrees(phi_t).round(3)}')
    if len(psi_e):
        # order estimates by Hungarian match for readable printing
        d = np.linalg.norm(np.stack([psi_t, phi_t],1)[:,None] -
                           np.stack([psi_e, phi_e],1)[None], axis=2)
        r, c = linear_sum_assignment(d)
        print(f'  est  ψ (AoA deg): {np.degrees(psi_e[c]).round(3)}')
        print(f'  est  φ (AoD deg): {np.degrees(phi_e[c]).round(3)}')
        print(f'  ψ error (deg):    {np.degrees(psi_e[c]-psi_t[r]).round(4)}')
        print(f'  φ error (deg):    {np.degrees(phi_e[c]-phi_t[r]).round(4)}')
    print(f'  → detected {ndet}/{ntot} sources within 1°')
    if good:
        print(f'  → RMSE over detected: {np.sqrt(np.mean(np.array(good)**2)):.4f}°')

print(f'\n{"═"*62}')
print('  ⚠️  এখানে error বড় হলে coordinate mapping ভুল (Bug 4 ধরনের)')
print('      GT থেকে recover করা angle ≈ true angle হওয়া বাধ্যতামূলক।')
print('      Expected: error < 0.3° (256-grid quantization only)')
print(f'{"═"*62}')

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 7 — CHECK #2: একই Y দিয়ে classical methods
# ══════════════════════════════════════════════════════════════
np.random.seed(7)
Y, gt, psi_t, phi_t, al = make_sample(L=3, P=16, nt=16, noiseless=True)

print('═'*66)
print('  ZERO NOISE — একই Y matrix, তিনটে classical method')
print('═'*66)
print(f'  true ψ (AoA deg): {np.degrees(psi_t).round(3)}')
print(f'  true φ (AoD deg): {np.degrees(phi_t).round(3)}')
print()

methods = [
    ('2D-DFT  (DFT-CEA)',  lambda Y, L: classical_dft(Y, L, NDFT=1024)),
    ('2D-DFT + SIC',       lambda Y, L: classical_dft_sic(Y, L, NDFT=1024)),
    ('2D-MUSIC (smoothed)',lambda Y, L: classical_music(Y, L, sub=12, Ngrid=361)),
]

for name, fn in methods:
    t0 = time.time()
    psi_e, phi_e = fn(Y, 3)
    dt = time.time() - t0
    ndet, ntot, good = evaluate(psi_e, phi_e, psi_t, phi_t)
    rmse = np.sqrt(np.mean(np.array(good)**2)) if good else float('nan')
    print(f'── {name}  ({dt*1000:.0f} ms)')
    if len(psi_e):
        d = np.linalg.norm(np.stack([psi_t, phi_t],1)[:,None] -
                           np.stack([psi_e, phi_e],1)[None], axis=2)
        r, c = linear_sum_assignment(d)
        print(f'     est ψ: {np.degrees(psi_e[c]).round(3)}   err: {np.degrees(psi_e[c]-psi_t[r]).round(3)}')
        print(f'     est φ: {np.degrees(phi_e[c]).round(3)}   err: {np.degrees(phi_e[c]-phi_t[r]).round(3)}')
    print(f'     detected {ndet}/{ntot}   RMSE = {rmse:.4f}°')
    print()

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 8 — Statistics: N trials, zero-noise + high SNR
# ══════════════════════════════════════════════════════════════
N_TRIALS = 300      # বাড়ালে বেশি সময় লাগবে (MUSIC ধীর)
L        = 3
RUN_MUSIC = False   # True করলে MUSIC-ও চলবে (অনেক ধীর)

scenarios = [('Zero noise', dict(noiseless=True)),
             ('SNR 25 dB',  dict(SNR=25)),
             ('SNR 20 dB',  dict(SNR=20)),
             ('SNR  0 dB',  dict(SNR=0))]

algos = [('GT-blob (upper bound)', None),
         ('2D-DFT  (DFT-CEA)',     lambda Y, L: classical_dft(Y, L, 1024)),
         ('2D-DFT + SIC',          lambda Y, L: classical_dft_sic(Y, L, 1024))]
if RUN_MUSIC:
    algos.append(('2D-MUSIC', lambda Y, L: classical_music(Y, L, 12, 361)))

results = {}
for sc_name, sc_kw in scenarios:
    np.random.seed(123)
    acc = {a[0]: dict(det=0, tot=0, errs=[]) for a in algos}
    t0 = time.time()
    for _ in range(N_TRIALS):
        Y, gt, psi_t, phi_t, _ = make_sample(L=L, P=16, nt=16, **sc_kw)
        for a_name, fn in algos:
            if fn is None:
                pk = peaks_from_img(gt, L)
                psi_e, phi_e = peaks2angles(pk)
            else:
                psi_e, phi_e = fn(Y, L)
            nd, nt_, gd = evaluate(psi_e, phi_e, psi_t, phi_t)
            acc[a_name]['det'] += nd
            acc[a_name]['tot'] += nt_
            acc[a_name]['errs'] += gd
    results[sc_name] = acc
    print(f'{sc_name}: done in {time.time()-t0:.1f}s')

# ── Table ──
print()
print('═'*78)
print(f'{"Scenario":<12} {"Method":<24} {"Pd":>8} {"RMSE (deg)":>12} {"detected":>14}')
print('─'*78)
for sc_name, _ in scenarios:
    for a_name, _ in algos:
        a = results[sc_name][a_name]
        pd_  = a['det']/a['tot'] if a['tot'] else float('nan')
        rmse = np.sqrt(np.mean(np.array(a['errs'])**2)) if a['errs'] else float('nan')
        print(f'{sc_name:<12} {a_name:<24} {pd_:>8.4f} {rmse:>12.4f} {a["det"]:>7}/{a["tot"]:<6}')
    print('─'*78)
print('═'*78)
print('Reference (paper Table II, ResNet):  SNR 20dB → Pd=0.925, RMSE=0.253°')
print('                                     SNR  0dB → Pd=0.643, RMSE=0.458°')
print('Reference (paper Table II, DFT-CEA): SNR 20dB → Pd=0.891, RMSE=0.227°')
print('                                     SNR  0dB → Pd=0.622, RMSE=0.437°')

In [ ]:
# Cell 9 — Visual: Y matrix, GT heatmap, DFT spectrum (একই sample)
np.random.seed(7)
Y, gt, psi_t, phi_t, _ = make_sample(L=3, P=16, nt=16, noiseless=True)

NDFT = 512
X = np.abs(np.fft.fft2(np.fft.ifft2(Y), s=(NDFT, NDFT)))

fig, ax = plt.subplots(1, 4, figsize=(17, 4))

im = ax[0].imshow(Y.real, cmap='RdBu_r'); plt.colorbar(im, ax=ax[0], fraction=.046)
ax[0].set_title('Y — Real (16×16)'); ax[0].set_xlabel('Tx beam p'); ax[0].set_ylabel('Rx beam q')

im = ax[1].imshow(np.abs(Y), cmap='hot'); plt.colorbar(im, ax=ax[1], fraction=.046)
ax[1].set_title('|Y| Magnitude'); ax[1].set_xlabel('Tx beam p'); ax[1].set_ylabel('Rx beam q')

im = ax[2].imshow(gt, cmap='viridis', origin='lower'); plt.colorbar(im, ax=ax[2], fraction=.046)
ax[2].set_title('Ground Truth heatmap (256×256)')
ax[2].set_xlabel('ωφ (AoD)'); ax[2].set_ylabel('ωψ (AoA)')

im = ax[3].imshow(np.fft.fftshift(X), cmap='inferno'); plt.colorbar(im, ax=ax[3], fraction=.046)
ax[3].set_title(f'Classical 2D-DFT spectrum ({NDFT}²)')
ax[3].set_xlabel('v = π cos φ'); ax[3].set_ylabel('u = π cos ψ')

plt.suptitle(f'Zero noise, L=3  |  true ψ={np.degrees(psi_t).round(1)}°, φ={np.degrees(phi_t).round(1)}°',
             fontsize=11, y=1.03)
plt.tight_layout()
plt.savefig('classical_sanity_test.png', dpi=130, bbox_inches='tight')
plt.show()
print('তিনটে bright spot দুই জায়গাতেই (GT আর DFT spectrum) একই position-এ থাকা উচিত।')